In [1]:
from pathlib import Path

import pandas as pd
import numpy as np

from sklearn.model_selection import StratifiedKFold

# Шлях до нашого проєкту
PROJECT_DIR = Path("/Users/glebsavelev/Desktop/da")

DATA_DIR = PROJECT_DIR / "results" / "data"
TABLES_DIR = PROJECT_DIR / "results" / "tables"

# Завантажуємо ТІЛЬКИ навчальні дані
X_train = pd.read_csv(
    DATA_DIR / "X_train.csv"
)

y_train = pd.read_csv(
    DATA_DIR / "y_train.csv"
).squeeze("columns")

# Перевірка даних
assert len(X_train) == len(y_train)
assert X_train.shape[1] == 8
assert set(y_train.unique()) == {0, 1}

print("=== НАВЧАЛЬНА ВИБІРКА ===")
print("Кількість записів:", len(X_train))
print("Кількість ознак:", X_train.shape[1])

print("\nРозподіл класів:")
print(y_train.value_counts())

# Створюємо стратифіковану кросвалідацію
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

fold_results = []

# Перевіряємо всі п'ять розбиттів
for fold_number, (train_idx, val_idx) in enumerate(
    cv.split(X_train, y_train),
    start=1
):

    y_fold_train = y_train.iloc[train_idx]
    y_fold_val = y_train.iloc[val_idx]

    # Навчальні та валідаційні індекси не перетинаються
    assert len(
        np.intersect1d(train_idx, val_idx)
    ) == 0

    fold_results.append({
        "Fold": fold_number,
        "Train_Size": len(train_idx),
        "Validation_Size": len(val_idx),
        "Train_Positive_Rate": y_fold_train.mean(),
        "Validation_Positive_Rate": y_fold_val.mean()
    })

fold_summary = pd.DataFrame(fold_results)

print("\n=== STRATIFIED 5-FOLD CV ===")
print(
    fold_summary
    .round(4)
    .to_string(index=False)
)

print("\nКросвалідацію успішно налаштовано.")
print("Тестова вибірка не використовувалася.")

=== НАВЧАЛЬНА ВИБІРКА ===
Кількість записів: 76916
Кількість ознак: 8

Розподіл класів:
diabetes
0    70130
1     6786
Name: count, dtype: int64

=== STRATIFIED 5-FOLD CV ===
 Fold  Train_Size  Validation_Size  Train_Positive_Rate  Validation_Positive_Rate
    1       61532            15384               0.0882                    0.0883
    2       61533            15383               0.0882                    0.0882
    3       61533            15383               0.0882                    0.0882
    4       61533            15383               0.0882                    0.0882
    5       61533            15383               0.0882                    0.0882

Кросвалідацію успішно налаштовано.
Тестова вибірка не використовувалася.


Крок 1. Створюємо Stratified 5-fold Cross-Validation

Крок 2. Кросвалідація Logistic Regression, Random Forest і Linear SVM

In [2]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV

from sklearn.model_selection import cross_validate

# 1. Групи ознак
numeric_features = [
    "age",
    "hypertension",
    "heart_disease",
    "bmi",
    "HbA1c_level",
    "blood_glucose_level"
]

categorical_features = [
    "gender",
    "smoking_history"
]

# 2. Попередня обробка
def make_preprocessor():

    numeric_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ])

    categorical_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder", OneHotEncoder(handle_unknown="ignore"))
    ])

    return ColumnTransformer([
        ("numeric", numeric_pipeline, numeric_features),
        ("categorical", categorical_pipeline, categorical_features)
    ])


# 3. Logistic Regression
lr_cv = Pipeline([
    ("preprocessor", make_preprocessor()),
    ("classifier", LogisticRegression(
        max_iter=1000,
        random_state=42
    ))
])

# 4. Random Forest
rf_cv = Pipeline([
    ("preprocessor", make_preprocessor()),
    ("classifier", RandomForestClassifier(
        n_estimators=100,
        max_depth=12,
        min_samples_leaf=2,
        random_state=42,
        n_jobs=-1
    ))
])

# 5. Linear SVM з коректним калібруванням
svm_inner = Pipeline([
    ("preprocessor", make_preprocessor()),
    ("classifier", LinearSVC(
        C=1.0,
        max_iter=5000,
        random_state=42
    ))
])

svm_cv = CalibratedClassifierCV(
    estimator=svm_inner,
    method="sigmoid",
    cv=3
)

# 6. Моделі
models_cv = {
    "Logistic Regression": lr_cv,
    "Random Forest": rf_cv,
    "Linear SVM": svm_cv
}

# 7. Однакові метрики для всіх моделей
scoring = {
    "Accuracy": "accuracy",
    "Precision": "precision",
    "Recall": "recall",
    "F1": "f1",
    "ROC_AUC": "roc_auc",
    "Average_Precision": "average_precision"
}

# 8. Кросвалідація
cv_fold_results = []

print("=== STRATIFIED 5-FOLD MODEL EVALUATION ===")

for name, model in models_cv.items():

    print(f"\nНавчання: {name}")

    scores = cross_validate(
        estimator=model,
        X=X_train,
        y=y_train,
        cv=cv,
        scoring=scoring,
        n_jobs=1,
        error_score="raise"
    )

    for fold in range(5):

        row = {
            "Model": name,
            "Fold": fold + 1
        }

        for metric in scoring:
            row[metric] = float(
                scores[f"test_{metric}"][fold]
            )

        cv_fold_results.append(row)

    print(f"{name}: 5 folds completed")


# 9. Результати кожного розбиття
cv_fold_summary = pd.DataFrame(cv_fold_results)

# 10. Середні значення і стандартні відхилення
metric_columns = list(scoring.keys())

cv_summary = (
    cv_fold_summary
    .groupby("Model", sort=False)[metric_columns]
    .agg(["mean", "std"])
)

cv_summary.columns = [
    f"{metric}_{stat}"
    for metric, stat in cv_summary.columns
]

cv_summary = cv_summary.reset_index()

# 11. Збереження результатів
TABLES_DIR.mkdir(parents=True, exist_ok=True)

cv_fold_summary.to_csv(
    TABLES_DIR / "cv_baseline_5fold_details.csv",
    index=False
)

cv_summary.to_csv(
    TABLES_DIR / "cv_baseline_5fold_summary.csv",
    index=False
)

print("\n=== ПІДСУМКОВІ CV-РЕЗУЛЬТАТИ ===")

print(
    cv_summary
    .round(4)
    .to_string(index=False)
)

print("\nКількість оцінених folds:", len(cv_fold_summary))
print("Збережено два CSV-файли.")
print("Тестова вибірка не використовувалася.")

=== STRATIFIED 5-FOLD MODEL EVALUATION ===

Навчання: Logistic Regression
Logistic Regression: 5 folds completed

Навчання: Random Forest
Random Forest: 5 folds completed

Навчання: Linear SVM
Linear SVM: 5 folds completed

=== ПІДСУМКОВІ CV-РЕЗУЛЬТАТИ ===
              Model  Accuracy_mean  Accuracy_std  Precision_mean  Precision_std  Recall_mean  Recall_std  F1_mean  F1_std  ROC_AUC_mean  ROC_AUC_std  Average_Precision_mean  Average_Precision_std
Logistic Regression         0.9587        0.0014          0.8644         0.0042       0.6310      0.0193   0.7293  0.0127        0.9619       0.0020                  0.8199                 0.0103
      Random Forest         0.9706        0.0010          0.9963         0.0027       0.6689      0.0105   0.8004  0.0077        0.9717       0.0014                  0.8766                 0.0059
         Linear SVM         0.9589        0.0016          0.8675         0.0052       0.6306      0.0211   0.7301  0.0139        0.9613       0.0021       

Попередні висновки:
- Random Forest має найвищі середні F1-score, ROC-AUC та Average Precision серед трьох перевірених алгоритмів.
- Random Forest також демонструє найменший розкид F1-score між п'ятьма folds.
- Logistic Regression і Linear SVM показують дуже близькі результати.
- Recall усіх трьох моделей за порогу 0,50 нижчий від 80%. 

Крок 3 — додаємо RBF-SVM із підбором гіперпараметрів
перевіряємо, чи здатна нелінійна SVM краще класифікувати діабетичний статус, ніж Linear SVM.

In [3]:
from sklearn.svm import SVC
from sklearn.model_selection import (
    train_test_split,
    RandomizedSearchCV
)
from sklearn.pipeline import Pipeline
import pandas as pd

# 1. Беремо лише частину TRAINING DATA
# для попереднього підбору параметрів
X_rbf_tune, _, y_rbf_tune, _ = train_test_split(
    X_train,
    y_train,
    train_size=15000,
    stratify=y_train,
    random_state=42
)

# 2. Нелінійна SVM
rbf_pipeline = Pipeline([
    ("preprocessor", make_preprocessor()),
    ("classifier", SVC(
        kernel="rbf",
        probability=False,
        cache_size=512
    ))
])

# 3. Простір пошуку
rbf_params = {
    "classifier__C": [0.1, 1, 10, 100],
    "classifier__gamma": [0.001, 0.01, 0.1, 1]
}

# 4. Підбір за Average Precision
rbf_search = RandomizedSearchCV(
    estimator=rbf_pipeline,
    param_distributions=rbf_params,
    n_iter=8,
    scoring=scoring,
    refit="Average_Precision",
    cv=cv,
    random_state=42,
    n_jobs=1,
    verbose=1,
    error_score="raise",
    return_train_score=False
)

print("=== RBF-SVM: HYPERPARAMETER SEARCH ===")
print("Навчальних записів:", len(X_rbf_tune))
print("Кількість комбінацій:", 8)
print("Кількість CV folds:", 5)

# 5. Запускаємо підбір
rbf_search.fit(X_rbf_tune, y_rbf_tune)

# 6. Результати
rbf_search_results = pd.DataFrame(
    rbf_search.cv_results_
)

columns = [
    "param_classifier__C",
    "param_classifier__gamma",
    "mean_test_Accuracy",
    "mean_test_Precision",
    "mean_test_Recall",
    "mean_test_F1",
    "mean_test_ROC_AUC",
    "mean_test_Average_Precision"
]

print("\n=== РЕЗУЛЬТАТИ ПОШУКУ ===")

print(
    rbf_search_results[columns]
    .sort_values(
        "mean_test_Average_Precision",
        ascending=False
    )
    .round(4)
    .to_string(index=False)
)

print("\n=== НАЙКРАЩІ ПАРАМЕТРИ ===")
print(rbf_search.best_params_)

print(
    "Найкраща середня Average Precision:",
    round(rbf_search.best_score_, 4)
)

# 7. Зберігаємо повні результати
output_path = (
    TABLES_DIR / "rbf_svm_preliminary_search.csv"
)

rbf_search_results.to_csv(
    output_path,
    index=False
)

print("\nЗбережено:", output_path.name)
print("Тестова вибірка не використовувалася.")

=== RBF-SVM: HYPERPARAMETER SEARCH ===
Навчальних записів: 15000
Кількість комбінацій: 8
Кількість CV folds: 5
Fitting 5 folds for each of 8 candidates, totalling 40 fits


/Users/glebsavelev/Desktop/da/venv/lib/python3.12/site-packages/sklearn/svm/_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
/Users/glebsavelev/Desktop/da/venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/glebsavelev/Desktop/da/venv/lib/python3.12/site-packages/sklearn/svm/_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
/Users/glebsavelev/Desktop/da/venv/lib/python3.12/site-packages/sklearn


=== РЕЗУЛЬТАТИ ПОШУКУ ===
 param_classifier__C  param_classifier__gamma  mean_test_Accuracy  mean_test_Precision  mean_test_Recall  mean_test_F1  mean_test_ROC_AUC  mean_test_Average_Precision
                10.0                    0.001              0.9595               0.9441            0.5752        0.7148             0.9606                       0.8217
                 1.0                    0.010              0.9593               0.9662            0.5578        0.7071             0.9549                       0.8214
                 0.1                    0.010              0.9538               0.9837            0.4845        0.6488             0.9588                       0.8198
               100.0                    0.100              0.9613               0.8935            0.6379        0.7436             0.9418                       0.8113
                 0.1                    0.001              0.9118               0.0000            0.0000        0.0000             0.9585 

Важлива деталь: конфігурація C=100, gamma=0.1 отримала вищий F1-score — 0,7436, але нижчу Average Precision — 0,8113. Це не суперечність: ми налаштували пошук на максимізацію саме Average Precision, як рекомендувала керівниця.
Також бачимо, що RBF-SVM чутлива до гіперпараметрів. За C=0.1, gamma=0.001 модель узагалі не прогнозувала позитивний клас при стандартному правилі класифікації, хоча ROC-AUC залишився 0,9585.
Поки не можна робити висновок, що RBF-SVM краща або гірша за Linear SVM чи Random Forest. Наші попередні три моделі оцінювалися на 76 916 навчальних записах, а пошук RBF-SVM проводився на підвибірці з 15 000.

Крок 4 — перевіряємо RBF-SVM на даних, які не брали участі в підборі

In [4]:
from sklearn.svm import SVC
from sklearn.pipeline import Pipeline
from sklearn.model_selection import (
    train_test_split,
    StratifiedKFold,
    cross_validate
)
from sklearn.metrics import make_scorer, precision_score

import pandas as pd

# 1. Відтворюємо попередній розподіл
# Ті самі 15 000 записів для підбору параметрів
# і решта даних для окремої оцінки

X_tune_check, X_rbf_eval, y_tune_check, y_rbf_eval = (
    train_test_split(
        X_train,
        y_train,
        train_size=15000,
        stratify=y_train,
        random_state=42
    )
)

# Переконуємося, що підвибірка підбору та сама
assert X_tune_check.index.equals(X_rbf_tune.index)

# Перевіряємо відсутність перетину
assert set(X_tune_check.index).isdisjoint(
    set(X_rbf_eval.index)
)

print("=== RBF-SVM: НЕЗАЛЕЖНА ВІД ПІДБОРУ CV-ОЦІНКА ===")
print("Записів для підбору:", len(X_tune_check))
print("Записів для CV-оцінки:", len(X_rbf_eval))

# 2. Зафіксовані гіперпараметри з попереднього пошуку
rbf_fixed = Pipeline([
    ("preprocessor", make_preprocessor()),
    ("classifier", SVC(
        kernel="rbf",
        C=10.0,
        gamma=0.001,
        cache_size=512
    ))
])

# 3. Стратифікована 5-fold CV
cv_rbf_eval = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

# 4. Ті самі шість метрик
# Для Precision явно задаємо поведінку,
# якщо модель не прогнозує позитивний клас

scoring_rbf = scoring.copy()

scoring_rbf["Precision"] = make_scorer(
    precision_score,
    zero_division=0
)

# 5. Оцінювання
scores_rbf = cross_validate(
    estimator=rbf_fixed,
    X=X_rbf_eval,
    y=y_rbf_eval,
    cv=cv_rbf_eval,
    scoring=scoring_rbf,
    n_jobs=1,
    error_score="raise"
)

# 6. Результати окремих folds
rbf_fold_rows = []

for fold in range(5):

    row = {
        "Model": "RBF SVM",
        "Fold": fold + 1,
        "C": 10.0,
        "gamma": 0.001
    }

    for metric in scoring_rbf:
        row[metric] = scores_rbf[
            f"test_{metric}"
        ][fold]

    rbf_fold_rows.append(row)

rbf_eval_details = pd.DataFrame(rbf_fold_rows)

# 7. Середні значення та стандартні відхилення
rbf_eval_summary = (
    rbf_eval_details[
        list(scoring_rbf.keys())
    ]
    .agg(["mean", "std"])
    .T
)

print("\n=== РЕЗУЛЬТАТИ RBF-SVM ===")
print(rbf_eval_summary.round(4))

# 8. Збереження
rbf_eval_details.to_csv(
    TABLES_DIR / "rbf_svm_5fold_eval_details.csv",
    index=False
)

rbf_eval_summary.to_csv(
    TABLES_DIR / "rbf_svm_5fold_eval_summary.csv"
)

print("\nЗбережено два CSV-файли.")
print("Тестова вибірка не використовувалася.")

=== RBF-SVM: НЕЗАЛЕЖНА ВІД ПІДБОРУ CV-ОЦІНКА ===
Записів для підбору: 15000
Записів для CV-оцінки: 61916

=== РЕЗУЛЬТАТИ RBF-SVM ===
                     mean     std
Accuracy           0.9596  0.0012
Precision          0.9439  0.0084
Recall             0.5762  0.0116
F1                 0.7156  0.0103
ROC_AUC            0.9586  0.0035
Average_Precision  0.8193  0.0103

Збережено два CSV-файли.
Тестова вибірка не використовувалася.


- Precision становить 94,39% — серед позитивних прогнозів моделі частка правильних досить висока.
- Recall становить лише 57,62% — модель пропускає значну частину позитивних записів за стандартним правилом класифікації.
- Average Precision = 0,8193 показує, що модель зберігає здатність ранжувати позитивні записи, навіть якщо стандартна межа класифікації дає недостатній Recall.
- Стандартні відхилення порівняно невеликі, але поки ми маємо лише одну 5-fold CV, а не repeated CV.
- Результат 0,8193 близький до 0,8217, отриманого під час підбору параметрів на 15 000 записах. Але це різні вибірки й різні етапи оцінювання.

2. Крок 5 — порівнюємо чотири моделі на однакових CV folds

In [5]:
from sklearn.model_selection import cross_validate
from sklearn.metrics import make_scorer, precision_score
import pandas as pd

print("=== ПОРІВНЯННЯ МОДЕЛЕЙ НА ОДНАКОВИХ CV FOLDS ===")

# 1. Використовуємо ті самі дані, що й для RBF-SVM
print("Кількість записів:", len(X_rbf_eval))
print("Кількість CV folds:", cv_rbf_eval.n_splits)

# 2. Ті самі метрики
scoring_matched = scoring.copy()

scoring_matched["Precision"] = make_scorer(
    precision_score,
    zero_division=0
)

# 3. Три базові моделі
baseline_models_matched = {
    "Logistic Regression": lr_cv,
    "Random Forest": rf_cv,
    "Linear SVM": svm_cv
}

matched_rows = []

# 4. Оцінюємо базові моделі
for name, model in baseline_models_matched.items():

    print(f"\nНавчання: {name}")

    scores = cross_validate(
        estimator=model,
        X=X_rbf_eval,
        y=y_rbf_eval,
        cv=cv_rbf_eval,
        scoring=scoring_matched,
        n_jobs=1,
        error_score="raise"
    )

    for fold in range(5):

        row = {
            "Model": name,
            "Fold": fold + 1
        }

        for metric in scoring_matched:

            row[metric] = float(
                scores[f"test_{metric}"][fold]
            )

        matched_rows.append(row)

    print(f"{name}: 5 folds completed")

# 5. Результати трьох моделей
baseline_matched_details = pd.DataFrame(
    matched_rows
)

# 6. Додаємо вже отримані результати RBF-SVM
metric_columns = list(scoring_matched.keys())

rbf_matched_details = rbf_eval_details[
    ["Model", "Fold"] + metric_columns
].copy()

# 7. Формуємо загальну таблицю
matched_cv_details = pd.concat(
    [
        baseline_matched_details,
        rbf_matched_details
    ],
    ignore_index=True
)

# 8. Перевіряємо повноту
assert len(matched_cv_details) == 20

assert (
    matched_cv_details
    .groupby("Model")["Fold"]
    .nunique()
    .eq(5)
    .all()
)

assert not matched_cv_details[
    metric_columns
].isna().any().any()

# 9. Обчислюємо середні та стандартні відхилення
matched_cv_summary = (
    matched_cv_details
    .groupby("Model", sort=False)[metric_columns]
    .agg(["mean", "std"])
)

matched_cv_summary.columns = [
    f"{metric}_{stat}"
    for metric, stat in matched_cv_summary.columns
]

matched_cv_summary = (
    matched_cv_summary
    .reset_index()
)

# 10. Зберігаємо результати
TABLES_DIR.mkdir(
    parents=True,
    exist_ok=True
)

matched_cv_details.to_csv(
    TABLES_DIR / "cv_matched_5fold_details.csv",
    index=False
)

matched_cv_summary.to_csv(
    TABLES_DIR / "cv_matched_5fold_summary.csv",
    index=False
)

# 11. Виводимо результати
print("\n=== ПОРІВНЯННЯ ЧОТИРЬОХ МОДЕЛЕЙ ===")

display_columns = [
    "Model",
    "Accuracy_mean",
    "Precision_mean",
    "Recall_mean",
    "F1_mean",
    "F1_std",
    "ROC_AUC_mean",
    "Average_Precision_mean"
]

print(
    matched_cv_summary[display_columns]
    .round(4)
    .to_string(index=False)
)

print("\nУсього результатів folds:", len(matched_cv_details))
print("Кількість моделей:", matched_cv_summary.shape[0])
print("Збережено два CSV-файли.")
print("Тестова вибірка не використовувалася.")

=== ПОРІВНЯННЯ МОДЕЛЕЙ НА ОДНАКОВИХ CV FOLDS ===
Кількість записів: 61916
Кількість CV folds: 5

Навчання: Logistic Regression
Logistic Regression: 5 folds completed

Навчання: Random Forest
Random Forest: 5 folds completed

Навчання: Linear SVM
Linear SVM: 5 folds completed

=== ПОРІВНЯННЯ ЧОТИРЬОХ МОДЕЛЕЙ ===
              Model  Accuracy_mean  Precision_mean  Recall_mean  F1_mean  F1_std  ROC_AUC_mean  Average_Precision_mean
Logistic Regression         0.9585          0.8617       0.6306   0.7282  0.0091        0.9617                  0.8193
      Random Forest         0.9706          0.9973       0.6689   0.8007  0.0076        0.9709                  0.8740
         Linear SVM         0.9587          0.8662       0.6297   0.7292  0.0101        0.9612                  0.8188
            RBF SVM         0.9596          0.9439       0.5762   0.7156  0.0103        0.9586                  0.8193

Усього результатів folds: 20
Кількість моделей: 4
Збережено два CSV-файли.
Тестова вибірка 

2. Крок 6 — додаємо CatBoost
CatBoost — це алгоритм градієнтного бустингу на деревах рішень. На відміну від наших попередніх моделей, він може безпосередньо працювати з категоріальними ознаками gender та smoking_history, без One-Hot Encoding. Це підтримується офіційною документацією CatBoost.

In [7]:
from catboost import CatBoostClassifier
from sklearn.model_selection import cross_validate
import pandas as pd

print("=== CATBOOST: STRATIFIED 5-FOLD CV ===")

# 1. Ті самі дані, що й для попередніх 4 моделей
assert len(X_rbf_eval) == 61916
assert len(X_rbf_eval) == len(y_rbf_eval)
assert not X_rbf_eval.isna().any().any()

# 2. CatBoost працює з категоріальними ознаками напряму
catboost_model = CatBoostClassifier(
    iterations=400,
    depth=6,
    learning_rate=0.05,
    l2_leaf_reg=3,
    loss_function="Logloss",
    random_seed=42,
    thread_count=4,
    verbose=False,
    allow_writing_files=False
)

# 3. Оцінювання на тих самих п'яти folds
scores_cb = cross_validate(
    estimator=catboost_model,
    X=X_rbf_eval,
    y=y_rbf_eval,
    cv=cv_rbf_eval,
    scoring=scoring_matched,
    params={
        "cat_features": [
            "gender",
            "smoking_history"
        ]
    },
    n_jobs=1,
    error_score="raise"
)

# 4. Результати кожного fold
catboost_rows = []

for fold in range(5):
    row = {
        "Model": "CatBoost",
        "Fold": fold + 1
    }

    for metric in scoring_matched:
        row[metric] = float(
            scores_cb[f"test_{metric}"][fold]
        )

    catboost_rows.append(row)

catboost_details = pd.DataFrame(catboost_rows)

# 5. Приєднуємо CatBoost до 4 попередніх моделей
five_model_details = pd.concat(
    [matched_cv_details, catboost_details],
    ignore_index=True
)

metric_columns = list(scoring_matched.keys())

assert len(five_model_details) == 25

# 6. Середні значення та стандартні відхилення
five_model_summary = (
    five_model_details
    .groupby("Model", sort=False)[metric_columns]
    .agg(["mean", "std"])
)

five_model_summary.columns = [
    f"{metric}_{stat}"
    for metric, stat in five_model_summary.columns
]

five_model_summary = five_model_summary.reset_index()

# 7. Зберігаємо нові файли, не перезаписуючи старі
five_model_details.to_csv(
    TABLES_DIR / "cv_matched_5fold_5models_details.csv",
    index=False
)

five_model_summary.to_csv(
    TABLES_DIR / "cv_matched_5fold_5models_summary.csv",
    index=False
)

# 8. Виводимо порівняння
display_columns = [
    "Model",
    "Accuracy_mean",
    "Precision_mean",
    "Recall_mean",
    "F1_mean",
    "F1_std",
    "ROC_AUC_mean",
    "Average_Precision_mean"
]

print("\n=== ПОРІВНЯННЯ П'ЯТИ МОДЕЛЕЙ ===")

print(
    five_model_summary[display_columns]
    .round(4)
    .to_string(index=False)
)

print("\nКількість моделей:", len(five_model_summary))
print("Кількість CV-результатів:", len(five_model_details))
print("Тестова вибірка не використовувалася.")
print("Результати збережено.")

=== CATBOOST: STRATIFIED 5-FOLD CV ===

=== ПОРІВНЯННЯ П'ЯТИ МОДЕЛЕЙ ===
              Model  Accuracy_mean  Precision_mean  Recall_mean  F1_mean  F1_std  ROC_AUC_mean  Average_Precision_mean
Logistic Regression         0.9585          0.8617       0.6306   0.7282  0.0091        0.9617                  0.8193
      Random Forest         0.9706          0.9973       0.6689   0.8007  0.0076        0.9709                  0.8740
         Linear SVM         0.9587          0.8662       0.6297   0.7292  0.0101        0.9612                  0.8188
            RBF SVM         0.9596          0.9439       0.5762   0.7156  0.0103        0.9586                  0.8193
           CatBoost         0.9704          0.9745       0.6828   0.8029  0.0076        0.9790                  0.8859

Кількість моделей: 5
Кількість CV-результатів: 25
Тестова вибірка не використовувалася.
Результати збережено.


Крок 7 — вибір порогів для двох сценаріїв
Тепер виконаємо рекомендацію керівниці щодо двох сценаріїв:
- Сценарій A — Max F1: знайти поріг, який забезпечує найвищий F1-score.
- Сценарій B — Screening: знайти поріг із найбільшим Precision за умови Recall ≥ 80% або Recall ≥ 90%.

In [8]:
import numpy as np
import pandas as pd

from sklearn.model_selection import (
    StratifiedKFold,
    cross_val_predict
)

from sklearn.metrics import (
    precision_recall_curve,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

print("=== THRESHOLD SELECTION: 5 MODELS ===")

# Тільки дані для підбору параметрів
assert len(X_rbf_tune) == 15000
assert len(X_rbf_tune) == len(y_rbf_tune)

cv_threshold = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

# Моделі та методи отримання оцінок
# Для RBF-SVM використовуємо decision_function,
# тобто поріг є порогом score, а не ймовірності.

threshold_models = {
    "Logistic Regression": (
        lr_cv, "predict_proba", None
    ),
    "Random Forest": (
        rf_cv, "predict_proba", None
    ),
    "Linear SVM": (
        svm_cv, "predict_proba", None
    ),
    "RBF SVM": (
        rbf_fixed, "decision_function", None
    ),
    "CatBoost": (
        catboost_model,
        "predict_proba",
        {"cat_features": ["gender", "smoking_history"]}
    )
}

threshold_rows = []

for name, (model, method, fit_params) in threshold_models.items():

    print(f"\nОбробка: {name}")

    # Out-of-fold оцінки:
    # для кожного запису прогноз отримано від моделі,
    # яка не навчалася на цьому записі.
    oof = cross_val_predict(
        estimator=model,
        X=X_rbf_tune,
        y=y_rbf_tune,
        cv=cv_threshold,
        method=method,
        params=fit_params,
        n_jobs=1
    )

    if method == "predict_proba":
        scores = oof[:, 1]
        standard_threshold = 0.50
    else:
        scores = oof
        standard_threshold = 0.0

    precision, recall, thresholds = precision_recall_curve(
        y_rbf_tune,
        scores
    )

    # Останній елемент precision/recall
    # не має відповідного порогу
    p = precision[:-1]
    r = recall[:-1]

    f1_values = np.divide(
        2 * p * r,
        p + r,
        out=np.zeros_like(p),
        where=(p + r) > 0
    )

    # Max F1
    best_f1_idx = int(np.argmax(f1_values))

    strategies = [
        ("Standard", standard_threshold),
        ("Max F1", thresholds[best_f1_idx])
    ]

    # Screening за двома вимогами Recall
    for target_recall in [0.80, 0.90]:

        valid = np.where(r >= target_recall)[0]

        if len(valid) == 0:
            continue

        # Максимальний Precision серед допустимих порогів
        best_precision = np.max(p[valid])

        candidates = valid[
            np.isclose(
                p[valid],
                best_precision,
                atol=1e-12
            )
        ]

        # За однакового Precision беремо вищий поріг
        selected_idx = int(candidates[-1])

        strategies.append((
            f"Recall >= {int(target_recall * 100)}%",
            thresholds[selected_idx]
        ))

    # Метрики для кожного обраного порогу
    for strategy, threshold in strategies:

        predictions = (
            scores >= threshold
        ).astype(int)

        tn, fp, fn, tp = confusion_matrix(
            y_rbf_tune,
            predictions,
            labels=[0, 1]
        ).ravel()

        threshold_rows.append({
            "Model": name,
            "Strategy": strategy,
            "Threshold": float(threshold),
            "Score_Type": (
                "Decision score"
                if method == "decision_function"
                else "Probability"
            ),
            "Precision": precision_score(
                y_rbf_tune,
                predictions,
                zero_division=0
            ),
            "Recall": recall_score(
                y_rbf_tune,
                predictions,
                zero_division=0
            ),
            "F1": f1_score(
                y_rbf_tune,
                predictions,
                zero_division=0
            ),
            "TP": int(tp),
            "FN": int(fn),
            "FP": int(fp),
            "TN": int(tn)
        })

    print(f"{name}: пороги визначено")

# Формуємо таблицю
threshold_selection_summary = pd.DataFrame(
    threshold_rows
)

print("\n=== ВИБРАНІ ПОРОГИ ===")

print(
    threshold_selection_summary[
        [
            "Model",
            "Strategy",
            "Threshold",
            "Precision",
            "Recall",
            "F1",
            "FN",
            "FP"
        ]
    ]
    .round(4)
    .to_string(index=False)
)

# Зберігаємо
output_path = (
    TABLES_DIR / "cv_threshold_selection_15000.csv"
)

threshold_selection_summary.to_csv(
    output_path,
    index=False
)

print("\nКількість конфігурацій:",
    len(threshold_selection_summary))

print("Збережено:", output_path.name)
print("Тестова вибірка не використовувалася.")

=== THRESHOLD SELECTION: 5 MODELS ===

Обробка: Logistic Regression
Logistic Regression: пороги визначено

Обробка: Random Forest
Random Forest: пороги визначено

Обробка: Linear SVM
Linear SVM: пороги визначено

Обробка: RBF SVM
RBF SVM: пороги визначено

Обробка: CatBoost
CatBoost: пороги визначено

=== ВИБРАНІ ПОРОГИ ===
              Model      Strategy  Threshold  Precision  Recall     F1  FN   FP
Logistic Regression      Standard     0.5000     0.8626  0.6266 0.7259 494  132
Logistic Regression        Max F1     0.4095     0.8085  0.6735 0.7348 432  211
Logistic Regression Recall >= 80%     0.1776     0.5861  0.8005 0.6767 264  748
Logistic Regression Recall >= 90%     0.0651     0.3869  0.9002 0.5412 132 1887
      Random Forest      Standard     0.5000     0.9955  0.6720 0.8023 434    4
      Random Forest        Max F1     0.4163     0.9795  0.6848 0.8060 417   19
      Random Forest Recall >= 80%     0.1678     0.6976  0.8005 0.7455 264  459
      Random Forest Recall >= 90% 

Тут уже видно головний компроміс: перехід від Max F1 до Recall близько 90% дозволив виявити на 300 позитивних записів більше, але призвів до 1 676 додаткових хибнопозитивних прогнозів.
Важливо: ці пороги вибрано на 15 000 навчальних записах за out-of-fold прогнозами. Ми ще не знаємо, чи забезпечать вони такий самий Recall на інших даних. Тому ці результати не слід подавати як незалежне підтвердження якості.

Крок 8 — перевіряємо вибрані пороги на окремих даних
Тепер застосуємо зафіксовані пороги до 61 916 записів, які не використовувалися для їхнього вибору.
Кожна модель знову пройде ті самі п'ять CV-розбиттів. Для кожного запису отримаємо прогноз від моделі, яка на ньому не навчалася.
Потім визначимо, скільки FP виникає при зменшенні FN, та чи справді досягаються цільові рівні Recall.

In [9]:
import numpy as np
import pandas as pd

from sklearn.model_selection import cross_val_predict
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix
)

print("=== THRESHOLD EVALUATION: 5 MODELS ===")

assert len(X_rbf_eval) == 61916
assert len(threshold_selection_summary) == 20

evaluation_rows = []

# Використовуємо моделі, визначені в кроці 7
for name, (model, method, fit_params) in threshold_models.items():

    print(f"\nОцінювання: {name}")

    # Out-of-fold прогнози на окремій вибірці
    oof_predictions = cross_val_predict(
        estimator=model,
        X=X_rbf_eval,
        y=y_rbf_eval,
        cv=cv_rbf_eval,
        method=method,
        params=fit_params,
        n_jobs=1
    )

    if method == "predict_proba":
        scores = oof_predictions[:, 1]
    else:
        scores = oof_predictions

    # Метрики ранжування не залежать від порогу
    roc_auc = roc_auc_score(y_rbf_eval, scores)

    average_precision = average_precision_score(
        y_rbf_eval, scores
    )

    # Пороги, вибрані в кроці 7
    selected_configs = threshold_selection_summary[
        threshold_selection_summary["Model"] == name
    ]

    assert len(selected_configs) == 4

    for _, config in selected_configs.iterrows():

        strategy = config["Strategy"]
        threshold = float(config["Threshold"])

        predictions = (
            scores >= threshold
        ).astype(int)

        tn, fp, fn, tp = confusion_matrix(
            y_rbf_eval,
            predictions,
            labels=[0, 1]
        ).ravel()

        recall = recall_score(
            y_rbf_eval,
            predictions,
            zero_division=0
        )

        target = {
            "Recall >= 80%": 0.80,
            "Recall >= 90%": 0.90
        }.get(strategy)

        evaluation_rows.append({
            "Model": name,
            "Strategy": strategy,
            "Threshold": threshold,
            "Score_Type": config["Score_Type"],
            "Accuracy": accuracy_score(
                y_rbf_eval, predictions
            ),
            "Precision": precision_score(
                y_rbf_eval,
                predictions,
                zero_division=0
            ),
            "Recall": recall,
            "F1": f1_score(
                y_rbf_eval,
                predictions,
                zero_division=0
            ),
            "ROC_AUC": roc_auc,
            "Average_Precision": average_precision,
            "TP": int(tp),
            "FN": int(fn),
            "FP": int(fp),
            "TN": int(tn),
            "Target_Met": (
                bool(recall >= target)
                if target is not None
                else None
            )
        })

    print(f"{name}: 4 стратегії оцінено")

# Загальна таблиця
threshold_eval_summary = pd.DataFrame(
    evaluation_rows
)

assert len(threshold_eval_summary) == 20

# Перевірка матриць помилок
assert (
    threshold_eval_summary[
        ["TP", "FN", "FP", "TN"]
    ].sum(axis=1) == len(y_rbf_eval)
).all()

print("\n=== РЕЗУЛЬТАТИ ОЦІНЮВАННЯ ПОРОГІВ ===")

print(
    threshold_eval_summary[
        [
            "Model",
            "Strategy",
            "Threshold",
            "Precision",
            "Recall",
            "F1",
            "FN",
            "FP",
            "Target_Met"
        ]
    ]
    .round(4)
    .to_string(index=False)
)

# Зберігаємо всі метрики
threshold_eval_summary.to_csv(
    TABLES_DIR / "cv_threshold_eval_61916.csv",
    index=False
)

# Порівнюємо screening із Max F1
max_f1_reference = (
    threshold_eval_summary[
        threshold_eval_summary["Strategy"] == "Max F1"
    ]
    .set_index("Model")[["FN", "FP"]]
    .rename(columns={
        "FN": "FN_MaxF1",
        "FP": "FP_MaxF1"
    })
)

tradeoff_summary = (
    threshold_eval_summary[
        threshold_eval_summary["Strategy"].isin([
            "Recall >= 80%",
            "Recall >= 90%"
        ])
    ]
    .copy()
    .join(max_f1_reference, on="Model")
)

tradeoff_summary["FN_Reduction"] = (
    tradeoff_summary["FN_MaxF1"]
    - tradeoff_summary["FN"]
)

tradeoff_summary["Additional_FP"] = (
    tradeoff_summary["FP"]
    - tradeoff_summary["FP_MaxF1"]
)

print("\n=== ЦІНА ПІДВИЩЕННЯ RECALL ===")

print(
    tradeoff_summary[
        [
            "Model",
            "Strategy",
            "Recall",
            "FN_Reduction",
            "Additional_FP",
            "Target_Met"
        ]
    ]
    .round(4)
    .to_string(index=False)
)

tradeoff_summary.to_csv(
    TABLES_DIR / "cv_threshold_tradeoff_61916.csv",
    index=False
)

print("\nПеревірено конфігурацій:",
      len(threshold_eval_summary))

print("Збережено два CSV-файли.")
print("Тестова вибірка не використовувалася.")

=== THRESHOLD EVALUATION: 5 MODELS ===

Оцінювання: Logistic Regression
Logistic Regression: 4 стратегії оцінено

Оцінювання: Random Forest
Random Forest: 4 стратегії оцінено

Оцінювання: Linear SVM
Linear SVM: 4 стратегії оцінено

Оцінювання: RBF SVM
RBF SVM: 4 стратегії оцінено

Оцінювання: CatBoost
CatBoost: 4 стратегії оцінено

=== РЕЗУЛЬТАТИ ОЦІНЮВАННЯ ПОРОГІВ ===
              Model      Strategy  Threshold  Precision  Recall     F1   FN   FP Target_Met
Logistic Regression      Standard     0.5000     0.8615  0.6306 0.7282 2018  554       None
Logistic Regression        Max F1     0.4095     0.8066  0.6694 0.7316 1806  877       None
Logistic Regression Recall >= 80%     0.1776     0.5740  0.7906 0.6651 1144 3206      False
Logistic Regression Recall >= 90%     0.0651     0.3863  0.9035 0.5412  527 7843       True
      Random Forest      Standard     0.5000     0.9973  0.6689 0.8007 1809   10       None
      Random Forest        Max F1     0.4163     0.9864  0.6769 0.8029 1765 

Ми бачимо важливий результат:
- Перехід від Max F1 до сценарію Recall ≥ 80% зменшив FN на 662, але збільшив FP на 2 329.
- Перехід від Max F1 до Recall ≥ 90% зменшив FN на 1 279, але збільшив FP на 6 966.
Особливо цікаво, що поріг, вибраний для Recall ≥ 80%, на нових даних забезпечив лише 79,06%.
За Random Forest уже видно, що стандартна конфігурація має Recall 66,89%, а Max F1 — 67,69%. 
Повні результати інших стратегій поки приховані скороченим виводом

Крок 9 — формуємо повну таблицю компромісів для п'яти моделей

In [10]:
import pandas as pd
import numpy as np

print("=== ПОРІВНЯННЯ SCREENING-СТРАТЕГІЙ ===")

# Завантажуємо результати попереднього кроку
results = pd.read_csv(
    TABLES_DIR / "cv_threshold_eval_61916.csv"
)

# Перевіряємо повноту
assert len(results) == 20

assert (
    results[["TP", "TN", "FP", "FN"]]
    .sum(axis=1)
    .eq(61916)
    .all()
)

# Беремо Max F1 як базову конфігурацію
baseline = (
    results[results["Strategy"] == "Max F1"]
    .set_index("Model")[["FN", "FP"]]
    .rename(columns={
        "FN": "FN_MaxF1",
        "FP": "FP_MaxF1"
    })
)

# Вибираємо сценарії Recall >= 80% та >= 90%
screening = results[
    results["Strategy"].isin([
        "Recall >= 80%",
        "Recall >= 90%"
    ])
].copy()

# Додаємо результати базової конфігурації
screening = screening.join(
    baseline,
    on="Model"
)

# Обчислюємо зменшення FN та збільшення FP
screening["FN_Reduction"] = (
    screening["FN_MaxF1"] - screening["FN"]
)

screening["Additional_FP"] = (
    screening["FP"] - screening["FP_MaxF1"]
)

# Скільки додаткових FP припадає
# на один усунутий FN
screening["FP_per_FN_saved"] = np.where(
    screening["FN_Reduction"] > 0,
    screening["Additional_FP"] / screening["FN_Reduction"],
    np.nan
)

# Виводимо повну таблицю
columns = [
    "Model",
    "Strategy",
    "Threshold",
    "Precision",
    "Recall",
    "F1",
    "FN",
    "FP",
    "FN_Reduction",
    "Additional_FP",
    "FP_per_FN_saved",
    "Target_Met"
]

print(
    screening[columns]
    .round(4)
    .to_string(index=False)
)

# Перевіряємо, які цілі Recall не досягнуті
failed = screening[
    screening["Target_Met"] == False
]

print("\n=== НЕДОСЯГНУТІ ЦІЛІ RECALL ===")

if failed.empty:
    print("Усі цільові рівні Recall досягнуті.")
else:
    print(
        failed[["Model", "Strategy", "Recall"]]
        .round(4)
        .to_string(index=False)
    )

# Зберігаємо таблицю
output_path = (
    TABLES_DIR / "screening_tradeoff_final.csv"
)

screening.to_csv(output_path, index=False)

print("\nЗбережено:", output_path.name)
print("Тестова вибірка не використовувалася.")

=== ПОРІВНЯННЯ SCREENING-СТРАТЕГІЙ ===
              Model      Strategy  Threshold  Precision  Recall     F1   FN   FP  FN_Reduction  Additional_FP  FP_per_FN_saved Target_Met
Logistic Regression Recall >= 80%     0.1776     0.5740  0.7906 0.6651 1144 3206           662           2329           3.5181      False
Logistic Regression Recall >= 90%     0.0651     0.3863  0.9035 0.5412  527 7843          1279           6966           5.4464       True
      Random Forest Recall >= 80%     0.1678     0.7087  0.7959 0.7498 1115 1787           650           1736           2.6708      False
      Random Forest Recall >= 90%     0.0790     0.4471  0.9160 0.6009  459 6187          1306           6136           4.6983       True
         Linear SVM Recall >= 80%     0.1713     0.5679  0.7946 0.6624 1122 3303           722           2487           3.4446      False
         Linear SVM Recall >= 90%     0.0678     0.3914  0.8958 0.5448  569 7610          1275           6794           5.3286      F

Це важливе спостереження: CatBoost — єдина з п'яти моделей, яка за попередньо обраними порогами досягла обох цільових рівнів Recall на окремих даних оцінювання.

Крок 10 — Data Quality Analysis

In [11]:
import numpy as np
import pandas as pd

print("=== DATA QUALITY ANALYSIS ===")

# 1. Дані лише з навчальної вибірки
quality_df = X_train.reset_index(drop=True).copy()
quality_df["diabetes"] = (
    y_train.reset_index(drop=True).astype(int)
)

assert len(quality_df) == 76916

TABLES_DIR.mkdir(parents=True, exist_ok=True)

# 2. Аналіз концентрації BMI = 27.32
quality_df["BMI_Group"] = np.where(
    np.isclose(quality_df["bmi"], 27.32),
    "BMI = 27.32",
    "Other BMI"
)

bmi_quality = (
    quality_df
    .groupby("BMI_Group")
    .agg(
        Count=("diabetes", "size"),
        Diabetes_Rate=("diabetes", "mean"),
        Age_Median=("age", "median"),
        HbA1c_Median=("HbA1c_level", "median"),
        Glucose_Median=("blood_glucose_level", "median"),
        Hypertension_Rate=("hypertension", "mean"),
        Heart_Disease_Rate=("heart_disease", "mean"),
        Smoking_NoInfo_Rate=(
            "smoking_history",
            lambda s: (s == "No Info").mean()
        )
    )
    .reset_index()
)

bmi_quality["Share_Percent"] = (
    100 * bmi_quality["Count"] / len(quality_df)
)

print("\n=== BMI = 27.32 VS OTHER BMI ===")
print(bmi_quality.round(4).to_string(index=False))

bmi_quality.to_csv(
    TABLES_DIR / "quality_bmi_2732_analysis.csv",
    index=False
)

# 3. Частоти лабораторних значень
for feature in ["HbA1c_level", "blood_glucose_level"]:

    frequency = (
        quality_df
        .groupby(feature)
        .agg(
            Count=("diabetes", "size"),
            Diabetes_Rate=("diabetes", "mean")
        )
        .reset_index()
        .sort_values(feature)
    )

    frequency["Share_Percent"] = (
        100 * frequency["Count"] / len(quality_df)
    )

    print(f"\n=== FREQUENCY: {feature} ===")
    print(frequency.round(4).to_string(index=False))

    frequency.to_csv(
        TABLES_DIR / f"quality_frequency_{feature}.csv",
        index=False
    )

# 4. Вікові значення біля верхньої межі
age_frequency = (
    quality_df.loc[quality_df["age"] >= 75, "age"]
    .value_counts()
    .sort_index()
    .reset_index()
)

age_frequency.columns = ["Age", "Count"]

print("\n=== AGE >= 75 ===")
print(age_frequency.to_string(index=False))

print(
    "Кількість записів age = 80:",
    int((quality_df["age"] == 80).sum())
)

age_frequency.to_csv(
    TABLES_DIR / "quality_age_75plus.csv",
    index=False
)

# 5. Історія куріння
smoking_quality = (
    quality_df
    .groupby("smoking_history")
    .agg(
        Count=("diabetes", "size"),
        Diabetes_Rate=("diabetes", "mean")
    )
    .reset_index()
)

smoking_quality["Share_Percent"] = (
    100 * smoking_quality["Count"] / len(quality_df)
)

print("\n=== SMOKING HISTORY ===")
print(smoking_quality.round(4).to_string(index=False))

smoking_quality.to_csv(
    TABLES_DIR / "quality_smoking_history.csv",
    index=False
)

print("\n=== ПІДСУМОК ===")
print("Проаналізовано записів:", len(quality_df))
print("Збережено 5 CSV-файлів.")
print("Тестова вибірка не використовувалася.")

=== DATA QUALITY ANALYSIS ===

=== BMI = 27.32 VS OTHER BMI ===
  BMI_Group  Count  Diabetes_Rate  Age_Median  HbA1c_Median  Glucose_Median  Hypertension_Rate  Heart_Disease_Rate  Smoking_NoInfo_Rate  Share_Percent
BMI = 27.32  17352         0.0700        44.0           5.8           140.0             0.0582              0.0360               0.4886        22.5597
  Other BMI  59564         0.0935        42.0           5.8           140.0             0.0838              0.0423               0.2993        77.4403

=== FREQUENCY: HbA1c_level ===
 HbA1c_level  Count  Diabetes_Rate  Share_Percent
         3.5   5852         0.0000         7.6083
         4.0   5768         0.0000         7.4991
         4.5   5880         0.0000         7.6447
         4.8   5855         0.0000         7.6122
         5.0   5747         0.0000         7.4718
         5.7   6458         0.0872         8.3962
         5.8   6410         0.0830         8.3338
         6.0   6373         0.0811         8.2857
 

За таблицею HbA1c видно незвичний зв'язок значень лабораторного показника з цільовою міткою:
- HbA1c від 3,5 до 5,0 у показаних групах — 0% позитивних записів.
- HbA1c = 6,8 — 100% позитивних записів.
- HbA1c = 7,0 — 100% позитивних записів.
- HbA1c = 7,5 — 100% позитивних записів.
Це підсилює нашу попередню підозру, що цільова мітка diabetes може бути тісно пов'язана з правилами формування лабораторних показників. Однак самих частот недостатньо, щоб довести витік цільової інформації.

Крок 11 — перевірка дублікатів і майже дублікатів
Ми вже видалили 3 854 повністю повторювані рядки з початкового датасету. Тепер перевіримо дві інші ситуації:
1. Чи є записи з абсолютно однаковими вхідними ознаками, але різними мітками diabetes.
2. Чи є записи, в яких сім ознак повністю збігаються, а BMI відрізняється не більше ніж на 0,1.

In [12]:
import pandas as pd
import numpy as np

print("=== DUPLICATE AND NEAR-DUPLICATE ANALYSIS ===")

# Працюємо лише з навчальними даними
df_check = X_train.reset_index(drop=True).copy()
df_check["diabetes"] = (
    y_train.reset_index(drop=True).astype(int)
)

feature_cols = X_train.columns.tolist()

# 1. Повністю однакові записи, включно з міткою
full_duplicates = int(
    df_check.duplicated().sum()
)

print("\nПовні дублікати після очищення:", full_duplicates)

# 2. Однакові вхідні ознаки, незалежно від мітки
feature_groups = (
    df_check
    .groupby(feature_cols, dropna=False)
    .agg(
        Count=("diabetes", "size"),
        Unique_Labels=("diabetes", "nunique")
    )
    .reset_index()
)

repeated_features = feature_groups[
    feature_groups["Count"] > 1
].copy()

conflicting_labels = repeated_features[
    repeated_features["Unique_Labels"] > 1
].copy()

print(
    "Груп з однаковими вхідними ознаками:",
    len(repeated_features)
)

print(
    "Груп із суперечливими мітками:",
    len(conflicting_labels)
)

# 3. Майже дублікати:
# усі ознаки, крім BMI, повністю однакові;
# різниця BMI більша за 0, але не перевищує 0.1

other_features = [
    col for col in feature_cols
    if col != "bmi"
]

ordered = (
    df_check
    .sort_values(other_features + ["bmi"])
    .copy()
)

ordered["BMI_Difference"] = (
    ordered
    .groupby(other_features, dropna=False)["bmi"]
    .diff()
)

near_duplicates = ordered[
    (ordered["BMI_Difference"] > 0)
    & (ordered["BMI_Difference"] <= 0.1)
].copy()

print(
    "Сусідніх пар потенційних майже дублікатів:",
    len(near_duplicates)
)

print(
    "Частка від навчальних записів, %:",
    round(
        100 * len(near_duplicates) / len(df_check),
        2
    )
)

# 4. Збереження результатів
duplicate_summary = pd.DataFrame([
    {
        "Indicator": "Training_records",
        "Value": len(df_check)
    },
    {
        "Indicator": "Full_duplicates",
        "Value": full_duplicates
    },
    {
        "Indicator": "Repeated_feature_groups",
        "Value": len(repeated_features)
    },
    {
        "Indicator": "Conflicting_label_groups",
        "Value": len(conflicting_labels)
    },
    {
        "Indicator": "Near_duplicate_adjacent_pairs",
        "Value": len(near_duplicates)
    }
])

duplicate_summary.to_csv(
    TABLES_DIR / "quality_duplicate_summary.csv",
    index=False
)

near_duplicates.head(100).to_csv(
    TABLES_DIR / "quality_near_duplicate_examples.csv",
    index=False
)

print("\n=== ПІДСУМОК ===")
print(duplicate_summary.to_string(index=False))

print("\nЗбережено два CSV-файли.")
print("Тестова вибірка не використовувалася.")

=== DUPLICATE AND NEAR-DUPLICATE ANALYSIS ===

Повні дублікати після очищення: 0
Груп з однаковими вхідними ознаками: 67
Груп із суперечливими мітками: 67
Сусідніх пар потенційних майже дублікатів: 351
Частка від навчальних записів, %: 0.46

=== ПІДСУМОК ===
                    Indicator  Value
             Training_records  76916
              Full_duplicates      0
      Repeated_feature_groups     67
     Conflicting_label_groups     67
Near_duplicate_adjacent_pairs    351

Збережено два CSV-файли.
Тестова вибірка не використовувалася.


Найважливіший результат — 67 груп із суперечливими мітками
Це означає, що в наборі є записи, у яких усі вісім вхідних характеристик однакові, але значення diabetes різне.
Наприклад, двом записам можуть відповідати однакові вік, BMI, HbA1c, рівень глюкози та інші ознаки, але одному присвоєно клас 0, а іншому — клас 1.
Для нашого дослідження це означає, що наявні вхідні ознаки не дозволяють однозначно визначити мітку для кожного запису.
Це може бути наслідком округлення, неповноти характеристик або особливостей формування датасету. 
Ми поки не знаємо справжньої причини, тому називати ці записи помилковими було б передчасно.
Також знайдено 351 сусідню пару потенційних майже дублікатів за нашим критерієм: сім ознак однакові,
 а BMI відрізняється не більше ніж на 0,1. Це кількість виявлених пар, а не доведена кількість повторних записів одних і тих самих пацієнтів.

Крок 12 — Sensitivity Analysis для BMI = 27,32
Тепер виконаємо ще одну конкретну рекомендацію керівниці.
Дослідницьке питання: чи змінюється якість моделей Random Forest та CatBoost, якщо виключити записи з BMI = 27,32 під час навчання?
Порівняємо два варіанти:
- Baseline: навчання на всіх записах відповідного CV-розбиття.
- Filtered: навчання без записів із BMI = 27,32.

In [13]:
import numpy as np
import pandas as pd

from sklearn.base import clone
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix
)
from catboost import CatBoostClassifier

print("=== BMI = 27.32: SENSITIVITY ANALYSIS ===")

# Використовуємо той самий набір для CV-порівняння
assert len(X_rbf_eval) == 61916
assert len(X_rbf_eval) == len(y_rbf_eval)

def create_sensitivity_model(name):

    if name == "Random Forest":
        return clone(rf_cv)

    if name == "CatBoost":
        return CatBoostClassifier(
            iterations=400,
            depth=6,
            learning_rate=0.05,
            l2_leaf_reg=3,
            loss_function="Logloss",
            random_seed=42,
            thread_count=4,
            verbose=False,
            allow_writing_files=False
        )

    raise ValueError(f"Unknown model: {name}")


sensitivity_rows = []

# Ті самі CV folds
for fold, (train_idx, val_idx) in enumerate(
    cv_rbf_eval.split(X_rbf_eval, y_rbf_eval),
    start=1
):

    X_fold_train = X_rbf_eval.iloc[train_idx]
    y_fold_train = y_rbf_eval.iloc[train_idx]

    X_fold_val = X_rbf_eval.iloc[val_idx]
    y_fold_val = y_rbf_eval.iloc[val_idx]

    # Видаляємо BMI = 27.32 ТІЛЬКИ з навчальної частини
    keep_mask = ~np.isclose(
        X_fold_train["bmi"].to_numpy(),
        27.32,
        rtol=0,
        atol=1e-8
    )

    X_filtered = X_fold_train.iloc[
        np.flatnonzero(keep_mask)
    ]

    y_filtered = y_fold_train.iloc[
        np.flatnonzero(keep_mask)
    ]

    removed_count = len(X_fold_train) - len(X_filtered)

    print(
        f"\nFold {fold}: "
        f"видалено {removed_count} навчальних записів"
    )

    for model_name in ["Random Forest", "CatBoost"]:

        for strategy in ["Baseline", "Filtered BMI"]:

            model = create_sensitivity_model(model_name)

            if strategy == "Baseline":
                X_fit_current = X_fold_train
                y_fit_current = y_fold_train
            else:
                X_fit_current = X_filtered
                y_fit_current = y_filtered

            # CatBoost: категоріальні ознаки передаємо у fit
            if model_name == "CatBoost":
                model.fit(
                    X_fit_current,
                    y_fit_current,
                    cat_features=[
                        "gender",
                        "smoking_history"
                    ]
                )
            else:
                model.fit(
                    X_fit_current,
                    y_fit_current
                )

            probabilities = model.predict_proba(
                X_fold_val
            )[:, 1]

            predictions = (
                probabilities >= 0.5
            ).astype(int)

            tn, fp, fn, tp = confusion_matrix(
                y_fold_val,
                predictions,
                labels=[0, 1]
            ).ravel()

            sensitivity_rows.append({
                "Model": model_name,
                "Strategy": strategy,
                "Fold": fold,
                "Train_Size": len(X_fit_current),
                "Removed_Records": (
                    removed_count
                    if strategy == "Filtered BMI"
                    else 0
                ),
                "Train_Positive_Rate": y_fit_current.mean(),
                "Accuracy": accuracy_score(
                    y_fold_val, predictions
                ),
                "Precision": precision_score(
                    y_fold_val,
                    predictions,
                    zero_division=0
                ),
                "Recall": recall_score(
                    y_fold_val,
                    predictions,
                    zero_division=0
                ),
                "F1": f1_score(
                    y_fold_val,
                    predictions,
                    zero_division=0
                ),
                "ROC_AUC": roc_auc_score(
                    y_fold_val,
                    probabilities
                ),
                "Average_Precision": average_precision_score(
                    y_fold_val,
                    probabilities
                ),
                "FN": int(fn),
                "FP": int(fp)
            })

            print(
                f"  {model_name} — {strategy}: завершено"
            )

# Детальні результати
sensitivity_details = pd.DataFrame(sensitivity_rows)

# Підсумок для кожної моделі та стратегії
sensitivity_summary = (
    sensitivity_details
    .groupby(["Model", "Strategy"], as_index=False)
    .agg(
        Train_Size_Mean=("Train_Size", "mean"),
        Train_Positive_Rate_Mean=(
            "Train_Positive_Rate", "mean"
        ),
        Accuracy_Mean=("Accuracy", "mean"),
        Precision_Mean=("Precision", "mean"),
        Recall_Mean=("Recall", "mean"),
        F1_Mean=("F1", "mean"),
        F1_Std=("F1", "std"),
        ROC_AUC_Mean=("ROC_AUC", "mean"),
        AP_Mean=("Average_Precision", "mean"),
        FN_Total=("FN", "sum"),
        FP_Total=("FP", "sum")
    )
)

print("\n=== SENSITIVITY ANALYSIS RESULTS ===")

print(
    sensitivity_summary
    .round(4)
    .to_string(index=False)
)

# Збереження
sensitivity_details.to_csv(
    TABLES_DIR / "bmi_sensitivity_5fold_details.csv",
    index=False
)

sensitivity_summary.to_csv(
    TABLES_DIR / "bmi_sensitivity_5fold_summary.csv",
    index=False
)

print("\nЗбережено два CSV-файли.")
print("Тестова вибірка не використовувалася.")

=== BMI = 27.32: SENSITIVITY ANALYSIS ===

Fold 1: видалено 11166 навчальних записів
  Random Forest — Baseline: завершено
  Random Forest — Filtered BMI: завершено
  CatBoost — Baseline: завершено
  CatBoost — Filtered BMI: завершено

Fold 2: видалено 11173 навчальних записів
  Random Forest — Baseline: завершено
  Random Forest — Filtered BMI: завершено
  CatBoost — Baseline: завершено
  CatBoost — Filtered BMI: завершено

Fold 3: видалено 11133 навчальних записів
  Random Forest — Baseline: завершено
  Random Forest — Filtered BMI: завершено
  CatBoost — Baseline: завершено
  CatBoost — Filtered BMI: завершено

Fold 4: видалено 11174 навчальних записів
  Random Forest — Baseline: завершено
  Random Forest — Filtered BMI: завершено
  CatBoost — Baseline: завершено
  CatBoost — Filtered BMI: завершено

Fold 5: видалено 11194 навчальних записів
  Random Forest — Baseline: завершено
  Random Forest — Filtered BMI: завершено
  CatBoost — Baseline: завершено
  CatBoost — Filtered BMI: зав

висновки
Random Forest: видалення BMI = 27,32 практично не вплинуло на F1-score. Зміна становить лише +0,00004.
CatBoost: F1-score збільшився з 0,8029 до 0,8048 (+0,0019). Покращення спостерігалося в усіх п'яти folds, але було дуже невеликим.
Водночас Average Precision і ROC-AUC обох моделей трохи зменшилися після фільтрації.

Крок 13 — контрольний експеримент із випадковим видаленням записів

In [14]:
import numpy as np
import pandas as pd
from sklearn.metrics import (
    f1_score,
    recall_score,
    roc_auc_score,
    average_precision_score
)

print("=== BMI SENSITIVITY: RANDOM REMOVAL CONTROL ===")

control_rows = []
random_seeds = [42, 123, 2026]

for fold, (train_idx, val_idx) in enumerate(
    cv_rbf_eval.split(X_rbf_eval, y_rbf_eval),
    start=1
):
    X_fold_train = X_rbf_eval.iloc[train_idx]
    y_fold_train = y_rbf_eval.iloc[train_idx]

    X_fold_val = X_rbf_eval.iloc[val_idx]
    y_fold_val = y_rbf_eval.iloc[val_idx]

    # Які записи видаляв попередній експеримент?
    bmi_removed = np.isclose(
        X_fold_train["bmi"].to_numpy(),
        27.32,
        rtol=0,
        atol=1e-8
    )

    train_labels = y_fold_train.to_numpy()

    # Точно відтворюємо кількість вилучень за класами
    remove_per_class = {
        label: int(
            np.sum(bmi_removed & (train_labels == label))
        )
        for label in [0, 1]
    }

    for repeat, seed in enumerate(random_seeds, start=1):

        rng = np.random.default_rng(seed + fold * 1009)

        random_removed = []

        for label in [0, 1]:

            candidates = np.flatnonzero(
                train_labels == label
            )

            selected = rng.choice(
                candidates,
                size=remove_per_class[label],
                replace=False
            )

            random_removed.extend(selected.tolist())

        keep = np.ones(len(X_fold_train), dtype=bool)
        keep[random_removed] = False

        X_random = X_fold_train.iloc[
            np.flatnonzero(keep)
        ]
        y_random = y_fold_train.iloc[
            np.flatnonzero(keep)
        ]

        assert int((~keep).sum()) == int(
            bmi_removed.sum()
        )

        for label in [0, 1]:
            assert int((y_random == label).sum()) == int(
                ((~bmi_removed) & (train_labels == label)).sum()
            )

        for name in ["Random Forest", "CatBoost"]:

            model = create_sensitivity_model(name)

            if name == "CatBoost":
                model.fit(
                    X_random,
                    y_random,
                    cat_features=["gender", "smoking_history"]
                )
            else:
                model.fit(X_random, y_random)

            probabilities = model.predict_proba(
                X_fold_val
            )[:, 1]

            predictions = (
                probabilities >= 0.5
            ).astype(int)

            control_rows.append({
                "Model": name,
                "Strategy": "Random matched removal",
                "Fold": fold,
                "Repeat": repeat,
                "Train_Size": len(X_random),
                "Train_Positive_Rate": y_random.mean(),
                "F1": f1_score(
                    y_fold_val, predictions
                ),
                "Recall": recall_score(
                    y_fold_val, predictions
                ),
                "ROC_AUC": roc_auc_score(
                    y_fold_val, probabilities
                ),
                "Average_Precision": average_precision_score(
                    y_fold_val, probabilities
                )
            })

        print(
            f"Fold {fold}, повторення {repeat}: завершено"
        )

# Результати випадкового видалення
control_details = pd.DataFrame(control_rows)

# Усереднюємо повторення в межах кожного fold
metric_cols = [
    "Train_Size",
    "Train_Positive_Rate",
    "F1",
    "Recall",
    "ROC_AUC",
    "Average_Precision"
]

control_by_fold = (
    control_details
    .groupby(["Model", "Strategy", "Fold"], as_index=False)
    [metric_cols]
    .mean()
)

# Додаємо попередні Baseline та Filtered BMI
previous = sensitivity_details[
    ["Model", "Strategy", "Fold"] + metric_cols
].copy()

comparison = pd.concat(
    [previous, control_by_fold],
    ignore_index=True
)

# Підсумок за однаковими п'ятьма folds
control_summary = (
    comparison
    .groupby(["Model", "Strategy"], as_index=False)
    .agg(
        Train_Size=("Train_Size", "mean"),
        Train_Positive_Rate=("Train_Positive_Rate", "mean"),
        F1_Mean=("F1", "mean"),
        F1_Std=("F1", "std"),
        Recall_Mean=("Recall", "mean"),
        ROC_AUC_Mean=("ROC_AUC", "mean"),
        AP_Mean=("Average_Precision", "mean")
    )
)

print("\n=== ПІДСУМКОВЕ ПОРІВНЯННЯ ===")

print(
    control_summary
    .round(5)
    .to_string(index=False)
)

# Зберігаємо результати
control_details.to_csv(
    TABLES_DIR / "bmi_random_control_details.csv",
    index=False
)

control_summary.to_csv(
    TABLES_DIR / "bmi_random_control_summary.csv",
    index=False
)

print("\nЗбережено два CSV-файли.")
print("Тестова вибірка не використовувалася.")

=== BMI SENSITIVITY: RANDOM REMOVAL CONTROL ===
Fold 1, повторення 1: завершено
Fold 1, повторення 2: завершено
Fold 1, повторення 3: завершено
Fold 2, повторення 1: завершено
Fold 2, повторення 2: завершено
Fold 2, повторення 3: завершено
Fold 3, повторення 1: завершено
Fold 3, повторення 2: завершено
Fold 3, повторення 3: завершено
Fold 4, повторення 1: завершено
Fold 4, повторення 2: завершено
Fold 4, повторення 3: завершено
Fold 5, повторення 1: завершено
Fold 5, повторення 2: завершено
Fold 5, повторення 3: завершено

=== ПІДСУМКОВЕ ПОРІВНЯННЯ ===
        Model               Strategy  Train_Size  Train_Positive_Rate  F1_Mean  F1_Std  Recall_Mean  ROC_AUC_Mean  AP_Mean
     CatBoost               Baseline     49532.8              0.08823  0.80289 0.00763      0.68277       0.97899  0.88594
     CatBoost           Filtered BMI     38364.8              0.09363  0.80478 0.00670      0.68460       0.97875  0.88565
     CatBoost Random matched removal     38364.8              0.09363  0

Головний науковий висновок
Для CatBoost видалення записів із BMI = 27,32 підвищило F1-score на 0,00190, але випадкове видалення аналогічної кількості записів також підвищило його на 0,00158.
Тобто різниця між двома способами видалення становить лише близько 0,00032 F1.
Для Random Forest різниця між цими способами також дуже мала — приблизно 0,00026 F1.
Отже, зараз у нас немає переконливих доказів, що саме значення BMI = 27,32 суттєво погіршує якість класифікації.
Невеликі зміни F1 можуть бути пов'язані зі скороченням навчальної вибірки, випадковими коливаннями або іншими відмінностями між записами.
Це хороший результат навіть без суттєвого покращення метрик: ми перевірили дослідницьку гіпотезу, а не просто видалили незручні дані.
Водночас три випадкові повторення — це обмежена перевірка, тому говорити про статистично доведену відсутність впливу також зарано.

Крок 14 — перевіряємо різницю між варіантами на кожному fold

In [15]:
import pandas as pd

print("=== BMI SENSITIVITY: PAIRED FOLD COMPARISON ===")

# Попередні результати
bmi_details = pd.read_csv(
    TABLES_DIR / "bmi_sensitivity_5fold_details.csv"
)

random_details = pd.read_csv(
    TABLES_DIR / "bmi_random_control_details.csv"
)

# Baseline і Filtered BMI для кожного fold
paired = bmi_details.pivot(
    index=["Model", "Fold"],
    columns="Strategy",
    values="F1"
).reset_index()

# Середній результат трьох випадкових повторень
random_by_fold = (
    random_details
    .groupby(["Model", "Fold"], as_index=False)["F1"]
    .mean()
    .rename(columns={"F1": "Random_Control"})
)

paired = paired.merge(
    random_by_fold,
    on=["Model", "Fold"],
    validate="one_to_one"
)

# Різниці F1 на однакових folds
paired["Filtered_minus_Baseline"] = (
    paired["Filtered BMI"] - paired["Baseline"]
)

paired["Random_minus_Baseline"] = (
    paired["Random_Control"] - paired["Baseline"]
)

paired["Filtered_minus_Random"] = (
    paired["Filtered BMI"] - paired["Random_Control"]
)

print("\n=== РІЗНИЦІ F1 ПО КОЖНОМУ FOLD ===")

print(
    paired.round(5).to_string(index=False)
)

# Підсумок
difference_summary = (
    paired
    .groupby("Model")[
        [
            "Filtered_minus_Baseline",
            "Random_minus_Baseline",
            "Filtered_minus_Random"
        ]
    ]
    .agg(["mean", "std"])
)

print("\n=== СЕРЕДНІ РІЗНИЦІ ===")

print(
    difference_summary.round(5).to_string()
)

# Збереження
paired.to_csv(
    TABLES_DIR / "bmi_paired_fold_comparison.csv",
    index=False
)

print("\nЗбережено: bmi_paired_fold_comparison.csv")
print("Тестова вибірка не використовувалася.")

=== BMI SENSITIVITY: PAIRED FOLD COMPARISON ===

=== РІЗНИЦІ F1 ПО КОЖНОМУ FOLD ===
        Model  Fold  Baseline  Filtered BMI  Random_Control  Filtered_minus_Baseline  Random_minus_Baseline  Filtered_minus_Random
     CatBoost     1   0.81316       0.81426         0.81416                  0.00110                0.00100                0.00010
     CatBoost     2   0.80497       0.80583         0.80431                  0.00085               -0.00066                0.00151
     CatBoost     3   0.79241       0.79544         0.79605                  0.00303                0.00363               -0.00060
     CatBoost     4   0.80432       0.80475         0.80644                  0.00042                0.00212               -0.00170
     CatBoost     5   0.79957       0.80365         0.80136                  0.00408                0.00179                0.00229
Random Forest     1   0.81260       0.81236         0.81476                 -0.00024                0.00215               -0.00239

висновок: ми не отримали переконливих доказів, що концентрація значень BMI = 27,32 сама по собі суттєво погіршує якість класифікації. Водночас незвичний розподіл залишається важливою характеристикою якості датасету.
Розкид відмінностей між folds більший за середню різницю між двома способами видалення. Тому заявляти про статистично доведену перевагу фільтрації BMI зараз не можна.